# MedAI Virtual Hospital Assistant Chatbot (Final Unique Version)

This notebook implements a **hybrid healthcare chatbot**:

- **Rule-based + state machine** for safe flows (appointments, medication reminders, emergency guidance)
- **Lightweight NLP retrieval** (TF‑IDF) for FAQ/service questions (no external API required)
- **Optional** local Generative model (DialoGPT) if you want a more “human” conversational style

> **Safety note:** This chatbot is for educational purposes and does **not** provide medical diagnosis.


In [ ]:
# If you run into missing packages, install them:
# !pip -q install scikit-learn dateparser transformers torch

import re
import json
from dataclasses import dataclass, field
from typing import Dict, Optional, List, Tuple

import dateparser
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Ready ✅')


## 1) Knowledge Base (MedAI services & FAQ)

We keep this **local** to make your work unique (instead of copying common templates). You can expand it.


In [ ]:
KB = [
    {
        "q": "What services do you offer?",
        "a": "MedAI provides: (1) appointment scheduling, (2) virtual consultations, (3) symptom guidance (non-diagnostic), (4) medication reminders, and (5) health education and lifestyle tips."
    },
    {
        "q": "Do you support diabetes risk screening?",
        "a": "Yes. MedAI supports early-risk screening through validated predictive models used by clinicians. The screening is not a final diagnosis; it flags potential risk and recommends follow-up with a healthcare professional."
    },
    {
        "q": "How do you protect privacy?",
        "a": "MedAI applies access control, data minimization, encryption in transit/at rest, and audit logging. Only authorized staff can access patient records, and data is used for care and service improvement under governance controls."
    },
    {
        "q": "Can I cancel or reschedule an appointment?",
        "a": "Yes. You can cancel or reschedule by providing your appointment reference, or by telling me the doctor name and date/time you booked."
    },
    {
        "q": "Do you provide emergency care?",
        "a": "MedAI is not an emergency service. If there is a medical emergency, contact local emergency services immediately."
    },
]

# Build a simple retriever for FAQ
kb_questions = [item["q"] for item in KB]
vectorizer = TfidfVectorizer(stop_words="english")
kb_matrix = vectorizer.fit_transform(kb_questions)

def kb_answer(user_text: str, min_score: float = 0.22) -> Optional[Tuple[str, float]]:
    """Return (answer, score) if a close FAQ match is found."""
    v = vectorizer.transform([user_text])
    sims = cosine_similarity(v, kb_matrix).ravel()
    idx = int(sims.argmax())
    score = float(sims[idx])
    if score >= min_score:
        return KB[idx]["a"], score
    return None

print('KB loaded ✅', f'({len(KB)} FAQ items)')


## 2) Safety Layer (Emergency red flags)

To make the chatbot **safer and more unique**, we add a rule-based layer that detects emergency symptoms and immediately advises urgent care.


In [ ]:
EMERGENCY_PATTERNS = [
    r"chest pain",
    r"difficulty breathing|shortness of breath",
    r"faint(ing)?|passed out",
    r"severe bleeding|bleeding a lot",
    r"stroke|face droop|slurred speech|one side weakness",
    r"suicidal|harm myself|kill myself",
]

def is_emergency(text: str) -> bool:
    t = text.lower()
    return any(re.search(p, t) for p in EMERGENCY_PATTERNS)

def emergency_message() -> str:
    return (
        "⚠️ This may be urgent. I can’t provide emergency medical help. "
        "Please contact local emergency services immediately or go to the nearest emergency department. "
        "If you’re not alone, ask someone nearby to help you."
    )

print('Safety layer ready ✅')


## 3) Conversation State Machine

We store the chat context (appointment booking, reminders, etc.). This helps your submission stand out versus simple if/else chatbots.


In [ ]:
@dataclass
class SessionState:
    mode: str = "idle"  # idle | booking | reminder
    booking: Dict[str, Optional[str]] = field(default_factory=lambda: {"doctor": None, "datetime": None, "reason": None})
    reminder: Dict[str, Optional[str]] = field(default_factory=lambda: {"medication": None, "time": None})
    history: List[Tuple[str, str]] = field(default_factory=list)  # (role, message)

state = SessionState()
print(state)


## 4) Intent Recognition (More robust than basic regex)

We use a weighted keyword approach + regex to detect intents.


In [ ]:
INTENTS = {
    "greet": ["hi", "hello", "hey", "good morning", "good evening"],
    "help": ["help", "what can you do", "menu", "options"],
    "services": ["services", "what do you offer", "features"],
    "book": ["book", "appointment", "schedule", "reserve", "consultation"],
    "cancel": ["cancel", "reschedule", "change appointment"],
    "reminder": ["remind", "reminder", "medicine", "medication"],
    "symptoms": ["symptom", "pain", "fever", "cough", "headache", "dizzy", "nausea"],
    "privacy": ["privacy", "data", "confidential", "security"],
    "exit": ["exit", "quit", "bye", "goodbye"],
}

def detect_intent(text: str) -> str:
    t = text.lower().strip()
    # Quick emergency gate
    if is_emergency(t):
        return "emergency"
    # Score intents
    scores = {k: 0 for k in INTENTS}
    for intent, kws in INTENTS.items():
        for kw in kws:
            if kw in t:
                scores[intent] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "unknown"

for x in ["Hi", "I want to book", "my chest pain", "privacy policy", "blahblah"]:
    print(x, '->', detect_intent(x))


## 5) Helper functions (date/time parsing, validation)


In [ ]:
def parse_datetime(user_text: str):
    # Prefer future dates when possible
    dt = dateparser.parse(
        user_text,
        settings={
            "PREFER_DATES_FROM": "future",
            "RETURN_AS_TIMEZONE_AWARE": False,
        },
    )
    return dt

def normalize_doctor_name(text: str) -> Optional[str]:
    t = text.strip()
    # Accept 'Dr Ahmed' or 'Ahmed'
    if len(t) < 2:
        return None
    if not re.search(r"[a-zA-Z]", t):
        return None
    if not t.lower().startswith("dr"):
        return "Dr " + t
    return t

def menu() -> str:
    return (
        "Here’s what I can help with:\n"
        "1) Book an appointment\n"
        "2) Reschedule/cancel an appointment\n"
        "3) Symptom guidance (non-diagnostic)\n"
        "4) Medication reminders\n"
        "5) MedAI services & privacy FAQs\n"
        "Type: 'book', 'symptoms', 'reminder', 'services', or 'privacy'."
    )


## 6) Core Chatbot Logic

This is the “final” function you will use for demos and screenshots.


In [ ]:
def bot_reply(user_text: str, state: SessionState) -> str:
    user_text = user_text.strip()
    if not user_text:
        return "Please type a message (e.g., 'book appointment', 'services')."

    # Emergency safety gate
    if is_emergency(user_text):
        state.mode = "idle"
        return emergency_message()

    # If we are in a flow, handle it first
    if state.mode == "booking":
        b = state.booking
        if b["doctor"] is None:
            doc = normalize_doctor_name(user_text)
            if doc is None:
                return "Please provide the doctor’s name (e.g., Dr Ahmed or Ahmed)."
            b["doctor"] = doc
            return f"Noted ✅ {doc}. What date/time do you prefer? (e.g., 'next Monday 10am')"

        if b["datetime"] is None:
            dt = parse_datetime(user_text)
            if dt is None:
                return "I couldn’t understand the date/time. Try like: 'tomorrow 5pm' or '12 March 9:30am'."
            b["datetime"] = dt.isoformat(sep=' ', timespec='minutes')
            return "Thanks. What is the reason for visit? (e.g., follow-up, checkup, diabetes screening)"

        if b["reason"] is None:
            reason = user_text[:80]
            b["reason"] = reason
            state.mode = "idle"
            # Generate a simple reference
            ref = f"MED-{abs(hash((b['doctor'], b['datetime'], b['reason']))) % 10_000_000:07d}"
            return (
                "✅ Appointment booked\n"
                f"• Doctor: {b['doctor']}\n"
                f"• Date/Time: {b['datetime']}\n"
                f"• Reason: {b['reason']}\n"
                f"• Reference: {ref}\n\n"
                "If you want to reschedule/cancel, type: 'reschedule' or 'cancel'."
            )

    if state.mode == "reminder":
        r = state.reminder
        if r["medication"] is None:
            r["medication"] = user_text[:60]
            return "Got it. When should I remind you? (e.g., 'every day 9pm' or 'tomorrow 8am')"
        if r["time"] is None:
            dt = parse_datetime(user_text)
            if dt is None:
                return "I couldn’t parse the time. Try: 'today 9pm' or 'tomorrow morning 8am'."
            r["time"] = dt.isoformat(sep=' ', timespec='minutes')
            state.mode = "idle"
            return (
                "✅ Reminder created (demo)\n"
                f"• Medication: {r['medication']}\n"
                f"• Time: {r['time']}\n\n"
                "(In a real system, this would be saved to the MedAI scheduling service.)"
            )

    # Otherwise, detect intent
    intent = detect_intent(user_text)

    if intent == "greet":
        return "Hello! I’m MedAI Assistant 🤖🏥\n" + menu()

    if intent == "help":
        return menu()

    if intent == "services":
        ans = kb_answer("What services do you offer?")
        return ans[0] if ans else "MedAI offers appointments, virtual consultations, and health education."

    if intent == "privacy":
        ans = kb_answer("How do you protect privacy?")
        return ans[0] if ans else "We follow strict privacy and security controls."

    if intent == "book":
        state.mode = "booking"
        state.booking = {"doctor": None, "datetime": None, "reason": None}
        return "Sure ✅ Let’s book your appointment. Which doctor would you like to see?"

    if intent == "reminder":
        state.mode = "reminder"
        state.reminder = {"medication": None, "time": None}
        return "Sure ✅ What medication should I remind you about?"

    if intent == "cancel":
        return "To cancel/reschedule, please share your appointment reference (e.g., MED-1234567) or tell me the doctor name + date/time."

    if intent == "symptoms":
        return (
            "I can help with *general guidance* (not diagnosis).\n"
            "1) Please describe your symptoms and how long you’ve had them.\n"
            "2) If you have severe symptoms (chest pain, trouble breathing, fainting, severe bleeding), seek urgent care immediately."
        )

    if intent == "exit":
        return "Goodbye 👋 Stay safe and take care."

    # Fallback: try FAQ retrieval
    faq = kb_answer(user_text)
    if faq:
        return f"(FAQ match score={faq[1]:.2f})\n" + faq[0]

    return (
        "I’m not fully sure I understood.\n"
        "Try: 'book appointment', 'services', 'privacy', 'symptoms', or 'reminder'."
    )


## 7) Run the Chatbot (Command-line demo)

Use this for screenshots in your assignment.


In [ ]:
def chat():
    print("MedAI Assistant: Hello! Welcome to MedAI.\n" + menu())
    while True:
        user = input("You: ")
        reply = bot_reply(user, state)
        print("MedAI Assistant:", reply)
        if detect_intent(user) == "exit":
            break

# Run:
# chat()


## 8) (Optional) Add a Generative Model Response (DialoGPT)

If you want a more natural conversation, you can enable a local generative model.
This is **optional** and you can keep it disabled for faster execution.


In [ ]:
# Optional: Enable a local generative component for the 'unknown' intent.
# Note: This requires downloading model weights.

USE_GENERATIVE = False

if USE_GENERATIVE:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    gen_model_name = "microsoft/DialoGPT-small"  # smaller and faster
    tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
    model = AutoModelForCausalLM.from_pretrained(gen_model_name)

    def generative_reply(user_text: str, max_len: int = 120) -> str:
        # Simple single-turn generation
        inputs = tokenizer.encode(user_text + tokenizer.eos_token, return_tensors='pt')
        output = model.generate(
            inputs,
            max_length=max_len,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_p=0.92,
            top_k=50,
        )
        text = tokenizer.decode(output[0], skip_special_tokens=True)
        # Return the part after the user text if possible
        return text[len(user_text):].strip() or text.strip()

    print('Generative module ready ✅')
else:
    print('Generative module disabled (recommended for speed) ✅')
